# G11.0 - Audit and setup

Discover and inspect every source artifact (GeoDAWN, fault labels, DEM tiles, template, optional G06/G07/G09/G10/M9/P6 evidence), fail fast on spatially unusable inputs, and run the synthetic integrity test suite before any expensive phase.

Nothing in this notebook trains on test labels, uploads to DrivenData or modifies source data. Every stage is resumable: re-running a cell reuses a valid stage record (`manifests/<stage>.json`).

In [ ]:
# Configuration overrides (everything else: configs/g11_multiscale.yaml). Paths are EXPECTED, verified by the audit.
OVERRIDES = {
    "data_root": "/content/drive/MyDrive/GEMS_2",
    "output_root": "outputs/G11_MULTISCALE_FAULT_DISCOVERY",
    "scratch_root": "/content/g11_scratch",
    "reuse_valid_cache": True,
}

In [ ]:
# Mount Drive, pin the source, install, load the config (no data is copied).
from google.colab import drive
drive.mount("/content/drive")
import subprocess, sys
from pathlib import Path
REPO, REF = "mybenkhadda/GEMS---drivendata", "g11-colab-production"
CANDIDATES = ["/content/GEMS", "/content/GEMS---drivendata", "/content/GEMS_G11"]
CODE = next((c for c in CANDIDATES if (Path(c) / ".git").exists()), CANDIDATES[0])
if not (Path(CODE) / ".git").exists():
    subprocess.run(["git", "clone", "--depth", "1", f"https://github.com/{REPO}.git", CODE], check=True)
subprocess.run(["git", "-C", CODE, "fetch", "--depth", "1", "origin", REF], check=True)
subprocess.run(["git", "-C", CODE, "checkout", "--detach", "FETCH_HEAD"], check=True)
if not (Path(CODE) / "gems_bench/g11/pipeline.py").is_file():
    raise RuntimeError(f"{REPO}@{REF} has no gems_bench/g11 - push the G11 package or set REF")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", CODE + "[colab]", "xgboost", "lightgbm"], check=True)
if CODE not in sys.path:
    sys.path.insert(0, CODE)
COMMIT = subprocess.check_output(["git", "-C", CODE, "rev-parse", "HEAD"], text=True).strip()
print("checkout", CODE, "commit", COMMIT)

from gems_bench.g11.config import load_config, Mode
from gems_bench.g11.pipeline import G11Pipeline
CFG = load_config(overrides=OVERRIDES)
P = G11Pipeline(CFG)
print("outputs:", P.ctx.root, "| config hash", P.ctx.cfg_hash)
print("Recommended entry point: GEMS_G11_MASTER_COLAB_EXECUTION.ipynb (this notebook runs one phase only)")

## Integrity tests (synthetic fixtures)

Must pass before INGEST / TRAIN. CPU only, a few minutes.

In [ ]:
r = subprocess.run([sys.executable, "-m", "pytest", f"{CODE}/gems_bench/g11/tests", "-q", "-p",
                    "no:cacheprovider"], capture_output=True, text=True)
print(r.stdout[-3000:])
assert r.returncode == 0, "integrity tests failed - stop here"

## Artifact audit

Required: GeoDAWN raster, fault-label raster, official template. Optional sources are recorded as ABSENT when missing (never zero-filled). NumPy predictions without a documented grid are UNLOCATED and excluded.

In [ ]:
summary = P.audit()
from IPython.display import Markdown, display
display(Markdown((P.ctx.d("audit") / "audit_report.md").read_text()))
display(Markdown((P.ctx.d("audit") / "dependency_graph.md").read_text()))

In [ ]:
import pandas as pd
reg = pd.read_parquet(P.ctx.d("audit") / "artifact_registry.parquet")
display(reg[[c for c in ("key", "path", "status", "alignment_confidence", "epsg", "width", "height") if c in reg]])